### Import packeges

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
from scipy.interpolate import griddata
from pathlib import Path
import h5py
import sys
from pathlib import Path

# Where am I running?
try:
    # Normal script
    here = Path(__file__).resolve().parent
except NameError:
    # Notebook / REPL
    here = Path.cwd()

phys_const_path = (here / '..' / 'phys_const').resolve()
sys.path.append(str(phys_const_path))

nsm_plots_path = (here / '..' / 'nsm_plots').resolve()
sys.path.append(str(nsm_plots_path))

nsm_plots_postproc = (here / '..' / 'nsm_instabilities').resolve()
sys.path.append(str(nsm_plots_postproc))

import phys_const as pc
import plot_functions as pf
import functions_angular_crossings as fac

### File path

In [ ]:
##################################################################################################################
# Simulation paths

direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_0'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-5'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-4'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-3'

parfile = '/plt00000_particles'
ncell = (1,1,1) # number of cells in each direction
domain = (1e5, 1e5, 1e5) # cm
num_particles_per_energy_bin = 92
num_energy_bins = 13
direct_interpolated_data = direct

##################################################################################################################

all_par_files = sorted(glob.glob(direct + '/plt*_particles'))
plt_numbers = [f.split('/')[-1].split('plt')[1].split('_particles')[0] for f in all_par_files]
plt_numbers = sorted(plt_numbers, key=lambda x: int(x))
all_par_files = [f'plt{plt_num}_particles' for plt_num in plt_numbers]

finaldir = direct.split('/')[-1]

# Calculate the volume of a single cell and the total volume
cellvolume = (domain[0] / ncell[0]) * (domain[1] / ncell[1]) * (domain[2] / ncell[2]) # cm^3
domainvolume = cellvolume * (ncell[0] * ncell[1] * ncell[2]) # cm^3

### Define cell index to be analyzed

In [ ]:
cell_index_i = 0
cell_index_j = 0
cell_index_k = 0

### Cell to be analized

In [ ]:
# Get the cell indices
cell_file_names = glob.glob(direct + parfile + '/cell_*_*_*')
cell_file_names = [file_name.split('/')[-1] for file_name in cell_file_names]
x_cell_ind = np.array([int(file_name.split('_')[1]) for file_name in cell_file_names])
y_cell_ind = np.array([int(file_name.split('_')[2]) for file_name in cell_file_names])
z_cell_ind = np.array([int((file_name.split('_')[3]).split('.')[0]) for file_name in cell_file_names])
cell_indices = np.array(list(zip(x_cell_ind, y_cell_ind, z_cell_ind)))
print('Number of cells:', len(cell_indices))
print(f'shape of cell_indices: {cell_indices.shape}')

# Get the cell indices for ix fix but iy and iz varying all available cells
x_idx_slice = cell_index_i
mask_yz_slice = cell_indices[:,0] == x_idx_slice # fixing the x index in this value
cell_indices_yz_slice = cell_indices[mask_yz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_yz_slice[:,1], cell_indices_yz_slice[:,2], color='b')
ax.scatter(cell_index_j, cell_index_k, color='r')
ax.set_xlabel('$i_y$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_x=${x_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iy fix but ix and iz varying all available cells
y_idx_slice = cell_index_j
mask_xz_slice = cell_indices[:,1] == y_idx_slice # fixing the y index in this value
cell_indices_xz_slice = cell_indices[mask_xz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xz_slice[:,0], cell_indices_xz_slice[:,2], color='b')
ax.scatter(cell_index_i, cell_index_k, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_y=${y_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iz fix but ix and iy varying all available cells
z_idx_slice = cell_index_k
mask_xy_slice = cell_indices[:,2] == z_idx_slice # fixing the z index in this value
cell_indices_xy_slice = cell_indices[mask_xy_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xy_slice[:,0], cell_indices_xy_slice[:,1], color='b')
ax.scatter(cell_index_i, cell_index_j, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_y$')
ax.set_title(f'Cell indices for fixed $i_z=${z_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Combine all cell indices from the three slices
# cell_indices_all = np.concatenate((cell_indices_yz_slice, cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.concatenate((cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.unique(cell_indices_all, axis=0)

### Getting temperature, electron fraction and density

In [ ]:
rho_ye_T_h5py = h5py.File(direct+'/rho_Ye_T.hdf5', 'r')

Temperature_MeV = np.array(rho_ye_T_h5py['/T_Mev'])[cell_index_i,cell_index_j,cell_index_k]
rho_g_cm3 = np.array(rho_ye_T_h5py['/rho_g|ccm'])[cell_index_i,cell_index_j,cell_index_k]
Ye = np.array(rho_ye_T_h5py['/Ye'])[cell_index_i,cell_index_j,cell_index_k]

print(f'rho_cons = {rho_g_cm3} # g/ccm')
print(f'T_cons = {Temperature_MeV} # MeV')
print(f'Ye_cons = {Ye} # n_electron - n_positron / n_barions')

neutrons_number_density_cm3 = rho_g_cm3 * (1.0 - Ye) / pc.PhysConst.Mp
protons_number_density_cm3 = rho_g_cm3 * Ye / pc.PhysConst.Mp

rho_ye_T_h5py.close()

### Compute ELN number density

In [ ]:
energybinsMeV = np.array([
    1, 3, 5.2382, 8.0097, 11.442, 15.691, 20.953, 27.468,
    35.536, 45.525, 57.895, 73.212, 92.178,
    # 115.66, 144.74, 180.75, 225.33, 280.54
])

energybinstopMeV = np.array([
    2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 102.67, 
    # 128.65, 160.83, 200.67, 250, 311.08
])

energybinsbottomMeV = np.array([
    0, 2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 
    # 102.67, 128.65, 160.83, 200.67, 250
])

nu_e_opacities_inv_cm = [1.1279864349828499e-08, 4.7428025628186796e-08, 1.321984832248114e-07, 3.294158114604995e-07, 7.528396092747342e-07, 1.5644452508683583e-06, 2.9487331095815796e-06, 5.1520838790595784e-06, 8.605493196949606e-06, 1.4043114431085709e-05, 2.262451577257334e-05, 3.6135649577238473e-05, 5.731900449884377e-05, 9.036764140607461e-05, 0.00014163060110313624, 0.00022059108455224182, 0.00034117744506685334, 0.0005234543477497763]
nu_x_opacities_inv_cm = [2.174662492044255e-10, 3.0391841308327543e-10, 4.466580398128411e-10, 6.371159236091845e-10, 8.969420647822444e-10, 1.2650799494987814e-09, 1.777602214007199e-09, 2.3794705306483863e-09, 3.149801121595646e-09, 4.208531959563822e-09, 5.307878395583215e-09, 6.9122806344757855e-09, 8.655465405273294e-09, 1.0987165360395685e-08, 1.3820008452372534e-08, 1.7330779319237295e-08, 2.1665700738968832e-08, 2.7031770968769257e-08]
nubar_e_opacities_inv_cm = [1.3566096423813977e-09, 4.4755275717913815e-09, 2.1546754963019424e-08, 5.47531006400814e-08, 1.1123883054511091e-07, 2.042363205567264e-07, 3.5482012940187117e-07, 5.920655243201202e-07, 9.526436748501559e-07, 1.4809548374270669e-06, 2.2289900021493824e-06, 3.2538736229147686e-06, 4.6123319631062746e-06, 6.3540654191625565e-06, 8.519538465802936e-06, 1.1153146580678764e-05, 1.4349950240540414e-05, 1.83608347661424e-05]

nu_e_opacities_inv_cm = np.array(nu_e_opacities_inv_cm)
nu_x_opacities_inv_cm = np.array(nu_x_opacities_inv_cm)
nubar_e_opacities_inv_cm = np.array(nubar_e_opacities_inv_cm)

nu_e_opacities_inv_cm = nu_e_opacities_inv_cm[0:13]
nu_x_opacities_inv_cm = nu_x_opacities_inv_cm[0:13]
nubar_e_opacities_inv_cm = nubar_e_opacities_inv_cm[0:13]

### Define function to get number density as a function of time

In [ ]:
def get_number_density_per_energy_bins(i, j, k, particlefile):

    parcellname = '/cell_' + str(i) + '_' + str(j) + '_' + str(k) + '.h5'

    particles_h5py = h5py.File(direct+particlefile+parcellname, 'r')
    N00_Re    = np.array(particles_h5py['/N00_Re'   ]) # particles
    N00_Rebar = np.array(particles_h5py['/N00_Rebar']) # particles
    N11_Re    = np.array(particles_h5py['/N11_Re'   ]) # particles
    N11_Rebar = np.array(particles_h5py['/N11_Rebar']) # particles
    N22_Re    = np.array(particles_h5py['/N22_Re'   ]) # particles
    N22_Rebar = np.array(particles_h5py['/N22_Rebar']) # particles
    time_s    = np.array(particles_h5py['/time'     ])[0] # seconds
    pupt      = np.array(particles_h5py['/pupt'     ]) # ergs
    pupx     = np.array(particles_h5py['/pupx'     ]) / pupt # unitless
    pupy     = np.array(particles_h5py['/pupy'     ]) / pupt # unitless
    pupz     = np.array(particles_h5py['/pupz'     ]) / pupt # unitless
    energyMeV = pupt / ( 1e6 * pc.CGSUnitsConst.eV ) # MeV     
    particles_h5py.close()

    n00_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n00_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n11_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3 
    n11_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3

    for t in range(num_energy_bins):
        energymask = (energyMeV >= energybinsbottomMeV[t]) & (energyMeV < energybinstopMeV[t])
        if np.sum(energymask) != num_particles_per_energy_bin:
            print(f"Warning: Energy bin {t} has {np.sum(energymask)} particles, expected {num_particles_per_energy_bin}")
        n00_Re_split_energy_bins   [t] = np.sum(N00_Re[energymask])    / cellvolume # particles / cm^3
        n00_Rebar_split_energy_bins[t] = np.sum(N00_Rebar[energymask]) / cellvolume # particles / cm^3
        n11_Re_split_energy_bins   [t] = np.sum(N11_Re[energymask])    / cellvolume # particles / cm^3
        n11_Rebar_split_energy_bins[t] = np.sum(N11_Rebar[energymask]) / cellvolume # particles / cm^3
        n22_Re_split_energy_bins   [t] = np.sum(N22_Re[energymask])    / cellvolume # particles / cm^3
        n22_Rebar_split_energy_bins[t] = np.sum(N22_Rebar[energymask]) / cellvolume # particles / cm^3

    return n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, time_s

### Plot the time evolution of the neutrino energy spectrum

In [ ]:

dE  = ( energybinstopMeV    - energybinsbottomMeV    ) * ( 1e6 * pc.CGSUnitsConst.eV )    # erg
dE3 = ( energybinstopMeV**3 - energybinsbottomMeV**3 ) * ( 1e6 * pc.CGSUnitsConst.eV )**3 # erg^3
E = ( energybinsMeV ) * ( 1e6 * pc.CGSUnitsConst.eV ) # erg
dOmega = 4.0 * np.pi

phase_space_factor = dOmega * dE * E**2  / pc.PhysConst.hc**3

In [ ]:
all_f00_Re_split_energy_bins    = []
all_f00_Rebar_split_energy_bins = []
all_f11_Re_split_energy_bins    = []
all_f11_Rebar_split_energy_bins = []
all_f22_Re_split_energy_bins    = []
all_f22_Rebar_split_energy_bins = []
tiempo_s = []

for dirpar in all_par_files:
    n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, t_s = get_number_density_per_energy_bins(cell_index_i, cell_index_j, cell_index_k,'/'+dirpar)
    all_f00_Re_split_energy_bins   .append(np.array(n00_Re_split_energy_bins)   / phase_space_factor)
    all_f00_Rebar_split_energy_bins.append(np.array(n00_Rebar_split_energy_bins)/ phase_space_factor)
    all_f11_Re_split_energy_bins   .append(np.array(n11_Re_split_energy_bins)   / phase_space_factor)
    all_f11_Rebar_split_energy_bins.append(np.array(n11_Rebar_split_energy_bins)/ phase_space_factor)
    all_f22_Re_split_energy_bins   .append(np.array(n22_Re_split_energy_bins)   / phase_space_factor)
    all_f22_Rebar_split_energy_bins.append(np.array(n22_Rebar_split_energy_bins)/ phase_space_factor)
    tiempo_s.append(t_s)

print('tiempo_s[-1]=',tiempo_s[-1])
all_f00_Re_split_energy_bins    = np.array(all_f00_Re_split_energy_bins)
all_f00_Rebar_split_energy_bins = np.array(all_f00_Rebar_split_energy_bins)
all_f11_Re_split_energy_bins    = np.array(all_f11_Re_split_energy_bins)
all_f11_Rebar_split_energy_bins = np.array(all_f11_Rebar_split_energy_bins)
all_f22_Re_split_energy_bins    = np.array(all_f22_Re_split_energy_bins)
all_f22_Rebar_split_energy_bins = np.array(all_f22_Rebar_split_energy_bins)
tiempo_s = np.array(tiempo_s)

print(f'energybinsMeV = {energybinsMeV}')
print(f'energybinsMeV.shape = {energybinsMeV.shape}')
print(f'all_f00_Re_split_energy_bins.shape = {all_f00_Re_split_energy_bins.shape}')
print(f'all_f00_Re_split_energy_bins[1].shape = {all_f00_Re_split_energy_bins[1].shape}')
print(f'tiempo_s.shape = {tiempo_s.shape}')

net_absorption_emission_n00_inv_s_inv_cm3 = np.zeros_like(tiempo_s)
net_absorption_emission_n00bar_inv_s_inv_cm3 = np.zeros_like(tiempo_s)

for i in range(len(tiempo_s)):
    net_absorption_emission_n00_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Re_split_energy_bins[i] - all_f00_Re_split_energy_bins[0]) * phase_space_factor )
    net_absorption_emission_n00bar_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Rebar_split_energy_bins[i] - all_f00_Rebar_split_energy_bins[0]) * phase_space_factor )

net_absorption_emission_protons_inv_s_inv_cm3   = +1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_neutrons_inv_s_inv_cm3  = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_electrons_inv_s_inv_cm3 = -1 * (                                                net_absorption_emission_n00_inv_s_inv_cm3 ) 
net_absorption_emission_positrons_inv_s_inv_cm3 = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3                                             ) 

print(f'net_absorption_emission_protons_inv_s_inv_cm3_ = {list(net_absorption_emission_protons_inv_s_inv_cm3)}')
print(f'tiempo_s_ = {list(tiempo_s)}')
print(f'...')

delta_tiempo_s = tiempo_s[1:] - tiempo_s[:-1]
cumulative_net_absorption_emission_protons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_protons_inv_s_inv_cm3[:-1] * delta_tiempo_s)
cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_neutrons_inv_s_inv_cm3[:-1] * delta_tiempo_s)

Ye_dot_inv_s = ( pc.PhysConst.Mp / rho_g_cm3 ) * ( net_absorption_emission_n00bar_inv_s_inv_cm3 -  net_absorption_emission_n00_inv_s_inv_cm3)
cumulative_Ye_inv_s = np.cumsum(Ye_dot_inv_s[:-1] * delta_tiempo_s)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_n00_inv_s_inv_cm3, label=r'$\dot{n}_{\nu_e}$')
ax.plot(tiempo_s, net_absorption_emission_n00bar_inv_s_inv_cm3, label=r'$\dot{n}_{\bar{\nu}_e}$')
ax.plot(tiempo_s, net_absorption_emission_electrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^-}$')
ax.plot(tiempo_s, net_absorption_emission_positrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^+}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
# ax.set_xlim(0, 0.005)
ax.set_title('Emission-absorption-driven changes\nwith flavor-conversion rates excluded.\nElectrons, positrons, neutrons,\nand protons are not integrated over time.\nAntineutrinos and neutrinos are integrated over time.\n')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=3, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n - n_{t_0} \,[\mathrm{cm}^{-3}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], neutrons_number_density_cm3 + cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.plot(tiempo_s[1:], protons_number_density_cm3 + cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n\,[\mathrm{{cm}}^{{-3}}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
# ax.set_xscale('log')
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, Ye_dot_inv_s)
ax.set_xlabel(r'$t\,[\mathrm{s}]$', fontsize=28)
ax.set_ylabel(r'$\dot{Y}_e\;[\mathrm{s}^{-1}]$', fontsize=28)
leg = ax.legend(framealpha=0.0, fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_Ye_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for cumulative change in Ye
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_Ye_inv_s, color='tab:blue', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e-Y_e^{t_0}$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_delta_Ye_cumulative.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for Ye evolution
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], Ye + cumulative_Ye_inv_s, color='tab:green', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_Ye_evolution.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
net_absorption_emission_protons_inv_s_inv_cm3_1emin5 = [0.0, -1.0309963309646358e+25, -3.537589474454929e+25, -1.1350485587161899e+26, -3.7176250886522504e+26, -1.2402947076638478e+27, -4.203513049647897e+27, -1.4438901515976385e+28, -5.014654678605111e+28, -1.7572074534195256e+29, -6.20198489866092e+29, -2.2016553733441116e+30, -7.85190516424343e+30, -2.8105794635816775e+31, -1.0089651057939819e+32, -3.630284482922594e+32, -1.3084334271088307e+33, -4.721237765894672e+33, -1.7038078907882906e+34, -6.132905203856571e+34, -2.182146436681224e+35, -7.446273231424822e+35, -2.2286433696555463e+36, -4.909816163324132e+36, -7.06967733397334e+36, -7.300788767629361e+36, -6.315818192767232e+36, -5.0485559980604137e+36, -3.913588249804638e+36, -3.0109091583188496e+36, -2.3264800067168454e+36, -1.819718281332481e+36, -1.4538748900256744e+36, -1.206227885075922e+36, -1.0744183975258048e+36, -1.0863375418751492e+36, -1.314380755542343e+36, -1.873590626097989e+36, -2.8312069625798866e+36, -3.983127674787219e+36, -4.819042672265873e+36, -4.9957736408676864e+36, -4.6220013797250775e+36, -3.986911993639141e+36, -3.3064899537440065e+36, -2.6861089253563446e+36, -2.1613212055045204e+36, -1.7344955338922557e+36, -1.3949329259066786e+36, -1.1281501899987565e+36, -9.1980444713906e+35, -7.572231390495092e+35, -6.298830038570992e+35, -5.294108442387378e+35, -4.493597287869324e+35, -3.8488879665688194e+35, -3.324221318248399e+35, -2.8933357677469433e+35, -2.5368301814745485e+35, -2.240124325457348e+35, -1.991981046248571e+35, -1.7834944193996903e+35, -1.607432575047952e+35, -1.4578314916538774e+35, -1.329752256026213e+35, -1.2191334757916608e+35, -1.1226908924751401e+35, -1.0378356577990235e+35, -9.625984267380979e+34, -8.955565319596882e+34, -8.357660147491102e+34, -7.827004572937588e+34, -7.361956559164547e+34, -6.963938734718274e+34, -6.636750251268908e+34, -6.38559026524769e+34, -6.215710664304076e+34, -6.130839246642964e+34, -6.131796625913226e+34, -6.215843035056174e+34, -6.377054088864436e+34, -6.6075324340042225e+34, -6.898860867799718e+34, -7.243184592886134e+34, -7.633666617664656e+34, -8.064523417488359e+34, -8.531115058679633e+34, -9.030483023701218e+34, -9.562355768339929e+34, -1.013018199717472e+35, -1.0741444397539904e+35, -1.1406532310957128e+35, -1.2135901915148192e+35, -1.293610659005385e+35, -1.3806287887006037e+35, -1.4737258942790614e+35, -1.5714595678588338e+35, -1.6724956136647555e+35, -1.776221133268543e+35, -1.8828799200052566e+35, -1.9929344731716884e+35, -2.1057619190643116e+35, -2.2182276371806747e+35, -2.3238875137762996e+35, -2.4133775921114252e+35, -2.476022266775756e+35, -2.5021438677575194e+35, -2.485336453300624e+35, -2.4241454710482287e+35, -2.3229027053245465e+35, -2.1915963537479737e+35, -2.0446269855373396e+35, -1.898438619112503e+35, -1.768476936328638e+35, -1.6663532520012957e+35, -1.5980374448938441e+35, -1.5634475904474324e+35, -1.5573533174477821e+35, -1.5712181151426146e+35, -1.5954015238737152e+35, -1.6210955164189202e+35, -1.6416156747059375e+35, -1.653073970385085e+35, -1.6547204400090812e+35, -1.6492074492301697e+35, -1.6427562856215805e+35]
tiempo_s_1emin5 = [0.0, 3.202215313919916e-05, 6.404430627839837e-05, 9.606645941760935e-05, 0.00012808861255681544, 0.0001601107656960004, 0.00019213291883518537, 0.00022415507197437034, 0.00025617722511353573, 0.00028819937825266866, 0.0003202215313918016, 0.0003522436845309345, 0.00038426583767006743, 0.00041628799080920036, 0.0004483101439483333, 0.0004803322970874662, 0.0005123544502265992, 0.0005443766033657321, 0.000576398756504865, 0.000608420909643998, 0.0006404430627831309, 0.0006724652159222638, 0.0007044873690613967, 0.0007365095222005297, 0.0007685316753396626, 0.0008005538284787955, 0.0008325759816179284, 0.0008645981347570614, 0.0008966202878961943, 0.0009286424410353272, 0.0009606645941744601, 0.0009926867473135932, 0.001024708900452726, 0.001056731053591859, 0.001088753206730992, 0.0011207753598701249, 0.0011527975130092578, 0.0011848196661483907, 0.0012168418192875236, 0.0012488639724266566, 0.0012808861255657895, 0.0013129082787049224, 0.0013449304318440553, 0.0013769525849831883, 0.0014089747381223212, 0.0014409968912614541, 0.001473019044400587, 0.00150504119753972, 0.001537063350678853, 0.0015690855038179858, 0.0016011076569571187, 0.0016331298100962517, 0.0016651519632353846, 0.0016971741163745175, 0.0017291962695136504, 0.0017612184226527834, 0.0017932405757919163, 0.0018252627289310492, 0.0018572848820701821, 0.001889307035209315, 0.001921329188348448, 0.001953351341487581, 0.001985373494626714, 0.002017395647765847, 0.00204941780090498, 0.002081439954044113, 0.0021134621071832458, 0.0021454842603223787, 0.0021775064134615116, 0.0022095285666006445, 0.0022415507197397775, 0.0022735728728789104, 0.0023055950260180433, 0.0023376171791571762, 0.002369639332296309, 0.002401661485435442, 0.002433683638574575, 0.002465705791713708, 0.002497727944852841, 0.002529750097991974, 0.0025617722511311067, 0.0025937944042702396, 0.0026258165574093726, 0.0026578387105485055, 0.0026898608636876384, 0.0027218830168267713, 0.0027539051699659043, 0.002785927323105037, 0.00281794947624417, 0.002849971629383303, 0.002881993782522436, 0.002914015935661569, 0.002946038088800702, 0.0029780602419398347, 0.0030100823950789677, 0.0030421045482181006, 0.0030741267013572335, 0.0031061488544963664, 0.0031381710076354994, 0.0031701931607746323, 0.0032022153139137652, 0.003234237467052898, 0.003266259620192031, 0.003298281773331164, 0.003330303926470297, 0.00336232607960943, 0.0033943482327485628, 0.0034263703858876957, 0.0034583925390268286, 0.0034904146921659615, 0.0035224368453050945, 0.0035544589984442274, 0.0035864811515833603, 0.0036185033047224932, 0.003650525457861626, 0.003682547611000759, 0.003714569764139892, 0.003746591917279025, 0.003778614070418158, 0.003810636223557291, 0.0038426583766964237, 0.0038746805298355566, 0.00390670268297469, 0.0039387248361138225, 0.003970746989252955, 0.004002769142392088]

net_absorption_emission_protons_inv_s_inv_cm3_1emin4 = [0.0, -1.5457226777472435e+25, -5.936394821989345e+25, -2.5602971613643656e+26, -9.751207464566664e+26, -3.7186121772809797e+27, -1.4160315586467618e+28, -5.3889821584038325e+28, -2.050859999648845e+29, -7.805315355184332e+29, -2.9707337107665463e+30, -1.1307021955272474e+31, -4.303664709784303e+31, -1.6380535338502532e+32, -6.234503036579937e+32, -2.372497108398642e+33, -9.022716359560483e+33, -3.423191206893052e+34, -1.2870324246025388e+35, -4.676324248236997e+35, -1.4985825875012285e+36, -3.216719013971845e+36, -3.3746316674534055e+36, -2.546292572918187e+36, -1.8941324564190718e+36, -1.4461549169807052e+36, -1.1840987150606608e+36, -1.015998123753715e+36, -1.0059789514122092e+36, -1.2348026423468098e+36, -1.6254518875487995e+36, -1.8407594463157146e+36, -1.6237666496711268e+36, -1.3487577925143674e+36, -1.1245691204848385e+36, -1.0268386445044858e+36, -1.0031388559016187e+36, -1.0105835691142747e+36, -1.0090426150803225e+36, -9.25158691315419e+35, -8.541815845514802e+35, -8.530710839127648e+35, -9.061561015009066e+35, -9.148104837439122e+35, -8.3160431691736e+35, -7.805350951875306e+35, -7.068059417400692e+35, -6.637001369637518e+35, -6.5236217905045745e+35, -6.225666474071244e+35, -6.22818070728778e+35, -6.093447635023903e+35, -5.704086863096949e+35, -5.599638650537988e+35, -6.2882265567532994e+35, -7.28487641104981e+35, -8.217302579480869e+35, -9.875468559696471e+35, -1.2554723609427382e+36, -1.4727020650871454e+36, -1.467495229902269e+36, -1.3788574239061525e+36, -1.0393926324190375e+36, -8.352935838235666e+35, -6.793803867263912e+35, -5.82742798289912e+35, -4.925810682649085e+35, -4.614104596614707e+35, -4.430322065609405e+35, -4.748124869608828e+35, -5.571644967229541e+35, -6.339420862149428e+35, -6.789055652798793e+35, -7.16603295745365e+35, -7.535121437676327e+35, -8.086868147880264e+35, -8.607633271911933e+35, -9.001220224216124e+35, -9.400637857137558e+35, -1.0103712975367058e+36, -1.0995994597490615e+36, -1.0994923866037425e+36, -1.0090124161900014e+36, -9.621944043802117e+35, -9.274511612210032e+35, -8.929391205725107e+35, -8.630521575173429e+35, -8.352133482712247e+35, -7.923110745534362e+35, -7.144646157535356e+35, -6.145096009631215e+35, -5.436389205471327e+35, -4.750502755727887e+35, -4.338816092399829e+35, -4.0556371498817855e+35, -3.881067254401365e+35, -3.462849588461679e+35, -3.094743045920115e+35, -2.9532321194236625e+35, -2.6176554503625354e+35, -2.237189523960672e+35, -1.9073257490176517e+35, -1.736388976295144e+35, -1.5847626211065472e+35, -1.429756038410292e+35, -1.26055985054877e+35, -1.129517859063613e+35, -1.0289700941512337e+35, -9.8924296229882e+34, -9.532529869482632e+34, -9.70226484082963e+34, -1.0171939949414617e+35, -1.0868197235155613e+35, -1.1436638362022762e+35, -1.1407348950401613e+35, -1.1314164111882526e+35, -1.1331933670503082e+35, -1.0780754142281669e+35, -9.547769334577166e+34, -8.771675001088016e+34, -8.777186973021121e+34, -9.709448076558723e+34, -1.0393867175638891e+35, -1.0315090655734668e+35, -1.0976990149617375e+35, -1.1941062612874804e+35]
tiempo_s_1emin4 = [0.0, 3.202215313921253e-05, 6.404430627850158e-05, 9.606645941779063e-05, 0.00012808861255703076, 0.0001601107656960596, 0.00019213291883508844, 0.00022415507197411728, 0.00025617722511295053, 0.00028819937825145896, 0.0003202215313899674, 0.0003522436845284758, 0.00038426583766698423, 0.00041628799080549266, 0.0004483101439440011, 0.0004803322970825095, 0.0005123544502218004, 0.0005443766033613497, 0.0005763987565008989, 0.0006084209096404482, 0.0006404430627799974, 0.0006724652159195467, 0.000704487369059096, 0.0007365095221986452, 0.0007685316753381945, 0.0008005538284777437, 0.000832575981617293, 0.0008645981347568422, 0.0008966202878963915, 0.0009286424410359408, 0.00096066459417549, 0.0009926867473160875, 0.0010247089004577184, 0.0010567310535993493, 0.0010887532067409803, 0.0011207753598826112, 0.0011527975130242421, 0.001184819666165873, 0.001216841819307504, 0.001248863972449135, 0.0012808861255907658, 0.0013129082787323968, 0.0013449304318740277, 0.0013769525850156586, 0.0014089747381572895, 0.0014409968912989205, 0.0014730190444405514, 0.0015050411975821823, 0.0015370633507238132, 0.0015690855038654442, 0.001601107657007075, 0.001633129810148706, 0.001665151963290337, 0.0016971741164319679, 0.0017291962695735988, 0.0017612184227152297, 0.0017932405758568607, 0.0018252627289984916, 0.0018572848821401225, 0.0018893070352817534, 0.0019213291884233844, 0.001953351341564986, 0.0019853734947024534, 0.002017395647839921, 0.0020494178009773886, 0.002081439954114856, 0.0021134621072523238, 0.0021454842603897913, 0.002177506413527259, 0.0022095285666647265, 0.002241550719802194, 0.0022735728729396617, 0.0023055950260771293, 0.002337617179214597, 0.0023696393323520645, 0.002401661485489532, 0.0024336836386269997, 0.0024657057917644673, 0.002497727944901935, 0.0025297500980394024, 0.00256177225117687, 0.0025937944043143376, 0.002625816557451805, 0.002657838710589273, 0.0026898608637267404, 0.002721883016864208, 0.0027539051700016756, 0.002785927323139143, 0.0028179494762766107, 0.0028499716294140783, 0.002881993782551546, 0.0029140159356890135, 0.002946038088826481, 0.0029780602419639487, 0.0030100823951014163, 0.003042104548238884, 0.0030741267013763515, 0.003106148854513819, 0.0031381710076512867, 0.0031701931607887542, 0.003202215313926222, 0.0032342374670636894, 0.003266259620201157, 0.0032982817733386246, 0.003330303926476092, 0.00336232607961356, 0.0033943482327510274, 0.003426370385888495, 0.0034583925390259626, 0.00349041469216343, 0.0035224368453008977, 0.0035544589984383653, 0.003586481151575833, 0.0036185033047133005, 0.003650525457850768, 0.0036825476109882357, 0.0037145697641257033, 0.003746591917263171, 0.0037786140704006385, 0.003810636223538106, 0.0038426583766755736, 0.0038746805298130412, 0.003906702682950391, 0.003938724836079532, 0.003970746989208673, 0.004002769142337814]

net_absorption_emission_protons_inv_s_inv_cm3_1emin3 = [0.0, -1.602924973113826e+25, -1.8804478511577484e+25, -8.98160992443511e+25, -9.003349636954381e+26, -3.767213007492544e+27, -1.4530489808298384e+28, -5.570342577455046e+28, -2.129041580382879e+29, -8.131890066072565e+29, -3.105646703199743e+30, -1.1860629769630989e+31, -4.529633130542573e+31, -1.7298711884181084e+32, -6.606068449714685e+32, -2.522230791242142e+33, -9.621510598076068e+33, -3.652097478704948e+34, -1.3294877402468511e+35, -3.329845885047933e+35, -3.369960177131998e+35, -2.626266607126275e+35, -2.1674902902382795e+35, -1.973405338565335e+35, -3.0831596909845367e+35, -2.7943368707576213e+35, -2.540445132729317e+35, -2.3127211696210976e+35, -2.2096694766566135e+35, -2.0389211527667074e+35, -2.200117099255246e+35, -2.3804595806345073e+35, -2.290022182403963e+35, -2.088197545615164e+35, -2.071217841513586e+35, -2.1444882034750665e+35, -1.985162532599227e+35, -1.9366958134603287e+35, -2.104440648571896e+35, -2.075417884348672e+35, -1.8421606029516145e+35, -2.1718420886886984e+35, -2.652871480234018e+35, -2.4160753437952226e+35, -2.2007109829922623e+35, -2.0812025911314604e+35, -2.1014277376901155e+35, -2.0637943482675e+35, -2.4871044161160092e+35, -3.8941512852470355e+35, -5.470637527000842e+35, -4.846795493722747e+35, -4.419598120048266e+35, -3.852527005797339e+35, -4.055589793069978e+35, -4.6562954504557144e+35, -4.668642005921919e+35, -5.162548111884615e+35, -4.757310089674287e+35, -4.1088799621639944e+35, -4.310776989712873e+35, -4.02724497813237e+35, -3.596196482544573e+35, -3.497377249842437e+35, -3.2464643390614133e+35, -3.1499181574422177e+35, -3.0609253560839374e+35, -3.135253314696544e+35, -2.9071824851617877e+35, -2.8217927828396174e+35, -3.27914753559995e+35, -3.4907443959274696e+35, -3.2560697307089686e+35, -3.079794292628814e+35, -2.5939025189375164e+35, -2.4879913259588187e+35, -2.376045965229328e+35, -2.3570434604826936e+35, -2.643787365277779e+35, -2.6768355115804122e+35, -2.637107806167196e+35, -2.5202576745039623e+35, -2.5100191738707446e+35, -2.3721658414627612e+35, -2.3448860487415004e+35, -2.4802806696267157e+35, -2.9267086986765733e+35, -4.5109523382836734e+35, -5.375583581122193e+35, -4.8579672351120346e+35, -4.320179872720114e+35, -3.777424859352365e+35, -3.79009139506865e+35, -3.719586708896467e+35, -3.376484506899756e+35, -3.0705360125974685e+35, -2.874535417541447e+35, -2.8086645361655982e+35, -2.597630330599932e+35, -2.487587009321166e+35, -2.3700226591185478e+35, -2.4268021141919745e+35, -2.568243822179928e+35, -2.351756743653953e+35, -2.547650495796102e+35, -3.2276189775242534e+35, -4.013039155503075e+35, -4.427592451655446e+35, -5.113164447174587e+35, -5.147345972670959e+35, -4.588348066937352e+35, -3.915082013614837e+35, -3.52452513720433e+35, -3.784160884089444e+35, -4.5600490176723704e+35, -5.792212755433772e+35, -6.2626444525519515e+35, -5.597789954501456e+35, -4.7365529108426204e+35, -4.016467586015969e+35, -3.454351519504009e+35, -2.8756192189452206e+35, -2.448030644195035e+35, -2.3914606051842273e+35, -2.3233626216029838e+35, -2.1996570310701255e+35]
tiempo_s_1emin3 = [0.0, 3.2022153139306275e-05, 6.404430627937595e-05, 9.606645941944563e-05, 0.00012808861255902626, 0.00016011076569649385, 0.00019213291883396144, 0.00022415507197142903, 0.0002561772251088966, 0.0002881993782463642, 0.0003202215313838318, 0.00035224368452129937, 0.00038426583765876696, 0.00041628799079623455, 0.00044831014393370214, 0.00048033229707116973, 0.0005123544502086373, 0.0005443766033461049, 0.0005763987564835725, 0.0006084209096210401, 0.0006404430627585077, 0.0006724652158959753, 0.0007044873690334429, 0.0007365095221709105, 0.000768531675308378, 0.0008005538284458456, 0.0008325759815833132, 0.0008645981347207808, 0.0008966202878582484, 0.000928642440995716, 0.0009606645941331836, 0.0009926867472706512, 0.0010247089004081188, 0.0010567310535455864, 0.001088753206683054, 0.0011207753598205215, 0.0011527975129579891, 0.0011848196660954567, 0.0012168418192329243, 0.001248863972370392, 0.0012808861255078595, 0.001312908278645327, 0.0013449304317827947, 0.0013769525849202623, 0.0014089747380577299, 0.0014409968911951974, 0.001473019044332665, 0.0015050411974701326, 0.0015370633506076002, 0.0015690855037450678, 0.0016011076568825354, 0.001633129810020003, 0.0016651519631574706, 0.0016971741162949382, 0.0017291962694324058, 0.0017612184225698733, 0.001793240575707341, 0.0018252627288448085, 0.0018572848819822761, 0.0018893070351197437, 0.0019213291882572113, 0.0019533513413949734, 0.0019853734945740743, 0.0020173956477531753, 0.0020494178009322762, 0.002081439954111377, 0.002113462107290478, 0.002145484260469579, 0.00217750641364868, 0.002209528566827781, 0.002241550720006882, 0.002273572873185983, 0.002305595026365084, 0.002337617179544185, 0.0023696393327232858, 0.0024016614859023867, 0.0024336836390814877, 0.0024657057922605886, 0.0024977279454396896, 0.0025297500986187905, 0.0025617722517978915, 0.0025937944049769924, 0.0026258165581560934, 0.0026578387113351943, 0.0026898608645142953, 0.0027218830176933963, 0.002753905170872497, 0.002785927324051598, 0.002817949477230699, 0.0028499716304098, 0.002881993783588901, 0.002914015936768002, 0.002946038089947103, 0.002978060243126204, 0.003010082396305305, 0.003042104549484406, 0.0030741267026635067, 0.0031061488558426077, 0.0031381710090217087, 0.0031701931622008096, 0.0032022153153799106, 0.0032342374685590115, 0.0032662596217381125, 0.0032982817749172134, 0.0033303039280963144, 0.0033623260812754153, 0.0033943482344545163, 0.0034263703876336172, 0.003458392540812718, 0.003490414693991819, 0.00352243684717092, 0.003554459000350021, 0.003586481153529122, 0.003618503306708223, 0.003650525459887324, 0.003682547613066425, 0.003714569766245526, 0.0037465919194246268, 0.0037786140726037277, 0.0038106362257828287, 0.0038426583789619296, 0.0038746805321410306, 0.003906702685318954, 0.003938724838414788, 0.0039707469915106226, 0.004002769144606457]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s_1emin3, net_absorption_emission_protons_inv_s_inv_cm3_1emin3, label=r"$10^{-3}$")
ax.plot(tiempo_s_1emin4, net_absorption_emission_protons_inv_s_inv_cm3_1emin4, label=r"$10^{-4}$")
ax.plot(tiempo_s_1emin5, net_absorption_emission_protons_inv_s_inv_cm3_1emin5, label=r"$10^{-5}$")
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
# ax.set_xlim(0.0, 0.005)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{Y}_e \, n_b = \dot{n}_p = \dot{n}_{\bar{\nu}_e} - \dot{n}_{\nu_e} \, [\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
ax.set_title('Ye time derivative.\n But it is not integrated over time in the \nsimulation since we use a frozen fluid.\n')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)